# Lab 9: GRPO Real com TRL

## Lab 9: GRPO Real com TRL

In [1]:
!pip install -q transformers torch trl datasets

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOTrainer, GRPOConfig

### 1. Reward function programática (verificador, Semana 9.5)

**Por que uma função, não um Reward Model:** GRPO não precisa de um
modelo separado treinado pra avaliar qualidade — só de uma função que
recebe as respostas geradas e devolve um número.

**Nota de metodologia (dois erros reais encontrados testando este lab):**
1) a primeira tentativa usava uma recompensa binária — "contém a palavra
'yes'?". Com um modelo tão pequeno, "yes" **nunca** apareceu em nenhuma
amostra — recompensa sempre 0, sem variância. 2) A segunda tentativa
recompensava diversidade de tokens (evitar repetição) — mas pesos
aleatórios do `tiny-gpt2` **já geram texto bem variado por padrão** (sem
tendência real a repetir), então essa recompensa ficou sempre perto de
1.0 — variância zero de novo, agora por excesso em vez de falta. Nos dois
casos, GRPO não tinha nada pra aprender (a vantagem relativa dentro do
grupo é zero quando todo mundo tira a mesma nota). A correção que
funcionou: medir o comprimento real gerado (que varia naturalmente,
~60-85 caracteres nesse setup) e recompensar proximidade de um alvo — um
critério que quase sempre distingue as amostras de um grupo.

In [2]:
TARGET_LENGTH = 70  # caracteres

def reward_target_length(completions, **kwargs):
    """Recompensa quão perto a completion fica de um comprimento-alvo —
    1.0 = exatamente no alvo, cai conforme se afasta. Contínua e com
    variância real neste setup (comprimento gerado varia naturalmente)."""
    rewards = []
    for completion in completions:
        text = completion if isinstance(completion, str) else str(completion)
        distance = abs(len(text) - TARGET_LENGTH)
        rewards.append(max(0.0, 1 - distance / TARGET_LENGTH))
    return rewards

# Teste rápido da reward function isolada
test_completions = ["x" * 70, "x" * 35, "x" * 140, "x" * 60]
print("Teste da reward function (alvo = 70 caracteres):")
for c, r in zip(test_completions, reward_target_length(test_completions)):
    print(f"  {len(c)} caracteres → reward={r:.2f}")

Teste da reward function (alvo = 70 caracteres):
  70 caracteres → reward=1.00
  35 caracteres → reward=0.50
  140 caracteres → reward=0.00
  60 caracteres → reward=0.86


**Resultado esperado:** `70 caracteres → reward=1.00` (exatamente no
alvo), `35 → reward=0.50` (metade do alvo, metade da distância máxima),
`140 → reward=0.00` (o dobro do alvo, distância = alvo inteiro), `60 →
reward≈0.86` — confirma a escala contínua funcionando.

### 2. Dataset de prompts (GRPO só precisa do prompt — a resposta é gerada)

**Diferença central pra SFT/DPO:** aqui não existe "resposta certa" no
dataset — só prompts. O modelo gera, a reward function avalia, o GRPO
ajusta a política pra gerar respostas com recompensa mais alta.

In [3]:
prompts = [
    {"prompt": "Is the sky blue?"},
    {"prompt": "Is water wet?"},
    {"prompt": "Is fire cold?"},
    {"prompt": "Is the sun a star?"},
] * 3  # repete pra ter exemplos suficientes pros grupos do GRPO

dataset = Dataset.from_list(prompts)
print(f"✓ {len(dataset)} prompts (sem resposta — o modelo gera, a reward function avalia)")

✓ 12 prompts (sem resposta — o modelo gera, a reward function avalia)


### 3. Medindo o reward médio ANTES do treino

In [4]:
import torch

MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def generate_batch(model, prompts_list, n_per_prompt=4, max_new_tokens=10):
    all_completions = []
    for p in prompts_list:
        inputs = tokenizer(p, return_tensors="pt")
        for _ in range(n_per_prompt):
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                  temperature=1.2, pad_token_id=tokenizer.eos_token_id)
            all_completions.append(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
    return all_completions

unique_prompts = ["Is the sky blue?", "Is water wet?", "Is fire cold?", "Is the sun a star?"]
before_completions = generate_batch(model, unique_prompts)
before_rewards = reward_target_length(before_completions)
print(f"Reward médio ANTES do GRPO: {sum(before_rewards)/len(before_rewards):.2f}")
print(f"Exemplo: {before_completions[0]!r}")

Reward médio ANTES do GRPO: 0.83
Exemplo: ' Daniel autonomy heir Money Participation autonomy credibilityimura Prob antibiotic'


**Resultado esperado:** um valor de reward em algum ponto entre 0 e 1 —
com `max_new_tokens=10`, o comprimento gerado tende a ficar naturalmente
na faixa de 60-85 caracteres, então não é incomum já começar relativamente
perto do alvo de 70 (reward ~0.8-0.9). O que importa pra esse lab não é o
valor absoluto, é que `reward_std` (visível no próximo passo) seja
diferente de zero — a variância é o que dá sinal pro GRPO aprender.

### 4. Treino real com GRPOTrainer

In [5]:
grpo_config = GRPOConfig(
    output_dir="./train_output",
    per_device_train_batch_size=4,
    num_generations=4,   # tamanho do "grupo" — Semana 9.4
    max_steps=15,
    learning_rate=1e-2,
    logging_steps=5,
    report_to="none",
    bf16=False, fp16=False,
    max_completion_length=10,
    temperature=1.2,      # precisa de amostragem com variação pra o grupo ter diversidade
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=reward_target_length,
    args=grpo_config,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
print("\n✓ GRPO concluído")

{'loss': '1.648e-07', 'grad_norm': '0.1642', 'learning_rate': '0.007333', 'num_tokens': '296', 'completions/mean_length': '10', 'completions/min_length': '10', 'completions/max_length': '10', 'completions/clipped_ratio': '1', 'completions/mean_terminated_length': '0', 'completions/min_terminated_length': '0', 'completions/max_terminated_length': '0', 'rewards/reward_target_length/mean': '0.9021', 'rewards/reward_target_length/std': '0.08176', 'reward': '0.9021', 'reward_std': '0.08176', 'frac_reward_zero_std': '0', 'entropy': '10.82', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '0.09574', 'epoch': '0.4167'}
{'loss': '-1.717e-07', 'grad_norm': '0.1625', 'learning_rate': '0.004', 'num_tokens': '596', 'completions/mean_length': '10', 'completions/min_length': '10', 'completions/max_length': '10', 'completions/clipped_ratio': '1', 'completions/mean_terminated_length': '0', 'compl

**Resultado esperado:** logs incluindo `reward` (a recompensa média por
step) e, principalmente, `reward_std` **maior que zero** — ao contrário
das duas tentativas anteriores (sempre 0 ou sempre 1), confirmando que o
GRPO finalmente tem variância real dentro de cada grupo pra calcular
vantagem relativa e aprender com ela.

### 5. Medindo o reward médio DEPOIS do treino

In [6]:
after_completions = generate_batch(trainer.model, unique_prompts)
after_rewards = reward_target_length(after_completions)

print("Exemplos DEPOIS do treino:")
for c in after_completions[:4]:
    print(f"  {c!r}")

print(f"\nReward médio ANTES:   {sum(before_rewards)/len(before_rewards):.2f}")
print(f"Reward médio DEPOIS: {sum(after_rewards)/len(after_rewards):.2f}")

Exemplos DEPOIS do treino:
  ' courtyard predators courtyard 236 rubbingSexual Singapore perhapsSexual Dreams'
  ' boilsSexualIsoblIs Redux soy clearer 236 grandchildren'
  ' workshopsozygMost factorspublic Singapore 1956publicSexual Singapore'
  'Mini Dreams Dreams Dreams grandchildren equateSexualpublicozyg Singapore'

Reward médio ANTES:   0.83
Reward médio DEPOIS: 0.89


**Resultado esperado:** o reward médio DEPOIS deveria ser igual ou maior
que ANTES — GRPO literalmente otimiza pra maximizar a reward function que
definimos, então chegar mais perto dos 70 caracteres-alvo é *exatamente*
o comportamento recompensado. Com poucos steps e um modelo tão pequeno, o
ganho pode ser modesto — o mecanismo importa mais que a magnitude aqui.

**Próximos passos:** Semana 10 aborda uma técnica completamente diferente
de melhorar um modelo pequeno: aprender com um modelo maior (Distillation),
em vez de RL ou preferências.